Anamoly Detection

Importing libraries and loading Dataset

In [51]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

#Loading datatset
df=pd.read_csv("../../datasets/data(2).csv")

#Display the head
df.head()

,node_id,proto_mean,seq_mean,stddev_mean,N_IN_Conn_P_SrcIP_mean,min_mean,state_number_mean,mean_mean,N_IN_Conn_P_DstIP_mean,drate_mean,srate_mean,max_mean,attack
0,192.168.100.1->192.168.100.147,0,4521.000000,0.0,6.000000,0.000055,2,0.000055,1,0.0,0.0,0.000055,0
1,192.168.100.1->192.168.100.148,0,4280.000000,0.0,6.000000,0.000058,2,0.000058,1,0.0,0.0,0.000058,0
2,192.168.100.1->192.168.100.149,0,5412.000000,0.0,6.000000,0.000111,2,0.000111,1,0.0,0.0,0.000111,0
3,192.168.100.1->192.168.100.150,0,4287.000000,0.0,6.000000,0.000065,2,0.000065,1,0.0,0.0,0.000065,0
4,192.168.100.1->192.168.100.27,0,5897.666667,0.0,2.333333,0.000371333333333333,2,0.000371333333333333,1,0.0,0.0,0.000371,0


In [52]:
#Dataset shape
df.shape

(156, 13)

In [53]:
#Dataset Info
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 156 entries, 0 to 155
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   node_id                 156 non-null    str    
 1   proto_mean              156 non-null    str    
 2   seq_mean                156 non-null    float64
 3   stddev_mean             156 non-null    float64
 4   N_IN_Conn_P_SrcIP_mean  156 non-null    float64
 5   min_mean                156 non-null    str    
 6   state_number_mean       152 non-null    str    
 7   mean_mean               156 non-null    str    
 8   N_IN_Conn_P_DstIP_mean  155 non-null    str    
 9   drate_mean              155 non-null    float64
 10  srate_mean              156 non-null    float64
 11  max_mean                156 non-null    float64
 12  attack                  156 non-null    int64  
dtypes: float64(6), int64(1), str(6)
memory usage: 16.0 KB


In [54]:
#Normal and attack records count
print(df["attack"].value_counts(dropna=False))

attack
0    124
1     32
Name: count, dtype: int64


In [55]:
#Remove node_id and attack
df=df.dropna(subset=["node_id","attack"]).copy()

In [56]:
#Convert attack labels to numeric values
df["attack"]=pd.to_numeric(df["attack"],errors="coerce")

In [57]:
# Remove any rows whose labels could not be converted
df = df.dropna(subset=["attack"]).copy()

In [58]:
df = df.reset_index(drop=True)

In [59]:
# Select all columns except the identifier and target
feature_columns=[
    col for col in df.columns
    if col not in ["node_id","attack"]
]

In [60]:
# Convert feature columns to numeric values
X_df=df[feature_columns].apply(pd.to_numeric,errors="coerce")

In [61]:
# Replace infinite values with missing values
X_df=X_df.replace([np.inf,-np.inf],np.nan)
X_df = X_df.fillna(X_df.median())
X_df = X_df.fillna(0)


In [62]:
# Scale the numerical features
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_df)

In [ ]:
# Convert features into a PyTorch tensor
x = torch.tensor(X_scaled, dtype=torch.float32)

In [64]:
# Convert attack labels into a tensor
y = torch.tensor(
    df["attack"].astype(int).to_numpy(),
    dtype=torch.float32
)

In [65]:
# Check the results
print("Feature tensor shape:", x.shape)
print("Label tensor shape:", y.shape)
print("Number of features:", x.shape[1])
print("Labels:", torch.unique(y))

Feature tensor shape: torch.Size([156, 11])
Label tensor shape: torch.Size([156])
Number of features: 11
Labels: tensor([0., 1.])


In [66]:
from torch_geometric.data import Data

#Storing the IP Addresses from each connection
node_ids=df["node_id"].astype(str).tolist()

# Remember the most recent node seen for each IP address
last_seen = {}

# Store the connections between nodes
edges = set()

for i,connection in enumerate(node_ids):
    # Split  the connection into source and destination IPs
    parts=connection.split("->")
    
    # Skip malfromed connection IDs
    if len(parts) !=2:
        continue
    
    source_ip=parts[0].strip()
    destination_ip=parts[1].strip()
    
    # Examine both IP Addresses
    for ip in set([source_ip,destination_ip]):
        # Ignore empty IP Addresses
        if not ip:
            continue
        
        # If this IP appeared before, connect the two nodes
        if ip in last_seen:
            previous_node=last_seen[ip]
            
            # Add an edge in both directions
            edges.add((previous_node,i))
            edges.add((i,previous_node))
            
        #Remember this node for the current IP
        last_seen[ip]=i
        
    #Convert the edges into a PyTorch tensor
    if edges:
        edge_index=torch.tensor(list(edges),dtype=torch.long).t().contiguous()
    else:
        edge_index = torch.empty((2, 0), dtype=torch.long)

In [67]:
# Create the graph
data=Data(x=x,edge_index=edge_index,y=y)

In [68]:
# Display the graph information
print(data)
print("Number of nodes:", data.num_nodes)
print("Number of edges:", data.num_edges)
print("Number of node features:", data.num_node_features)

Data(x=[156, 11], edge_index=[2, 438], y=[156])
Number of nodes: 156
Number of edges: 438
Number of node features: 11


In [69]:
from sklearn.model_selection import train_test_split

#  Create an index for every node
indices= np.arange(data.num_nodes)

# Split node indices while preserving the class proportions
train_x,test_x=train_test_split(indices,test_size=0.2,random_state=42,stratify=y.numpy())

In [70]:
# Convert indices to PyTorch tensors
train_x = torch.tensor(train_x, dtype=torch.long)
test_x = torch.tensor(test_x, dtype=torch.long)

print("Training nodes:", len(train_x))
print("Testing nodes:", len(test_x))

Training nodes: 124
Testing nodes: 32


In [78]:
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class GCN(nn.Module):
    def __init__(self,input_features,hidden_features=16):
            super().__init__()
            
            self.conv1=GCNConv(input_features,hidden_features)
            self.conv2=GCNConv(hidden_features,1)
            
    def forward(self,x,edge_index):
        # Learn from node features and neighboring nodes
        x=self.conv1(x,edge_index)
        
        #Applying ReLu activation
        x=F.relu(x)
        
        # Produce one score for each node
        x = self.conv2(x, edge_index)
        
        # Change shape from (nodes, 1) to (nodes,)
        return x.squeeze(-1)

In [79]:
#Creating model
model = GCN(
    input_features=data.num_node_features,
    hidden_features=16
)

print(model)

GCN(
  (conv1): GCNConv(11, 16)
  (conv2): GCNConv(16, 1)
)


In [80]:
# Loss function for binary classification
criterion = torch.nn.BCEWithLogitsLoss()

# Optimizer that updates the model's parameters
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

In [81]:
for epoch in range(1,11):
    #Training model
    model.train()
 
    optimizer.zero_grad()
    
    # Predict scores for all nodes
    output=model(data.x,data.edge_index)
    
    # Calculate loss using training nodes only
    loss = criterion(
        output[train_x],
        data.y[train_x]
    )
    
    loss.backward()
    optimizer.step()
    
    if epoch % 10 == 0:
        print(
            f"Epoch {epoch:02d}, Loss: {loss.item():.4f}"
        )

Epoch 10, Loss: 0.4103


In [83]:
#Model Evaluation
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,classification_report

model.eval()

with torch.no_grad():

    output = model(data.x, data.edge_index)
    
    # Get scores for all nodes
    output = model(data.x, data.edge_index)

    # Convert scores into probabilities
    probabilities = torch.sigmoid(output)

    # Convert probabilities into 0 or 1 predictions
    predictions = (probabilities >= 0.5).int()

# Get the actual labels for the test nodes
y_true = data.y[test_x].int().numpy()

# Get predictions for the test nodes
y_pred = predictions[test_x].numpy()

# Display evaluation results
print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, zero_division=0))
print("Recall:", recall_score(y_true, y_pred, zero_division=0))
print("F1-score:", f1_score(y_true, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    target_names=["Normal", "Attack"],
    zero_division=0
))

Accuracy: 0.875
Precision: 1.0
Recall: 0.42857142857142855
F1-score: 0.6

Confusion Matrix:
[[25  0]
 [ 4  3]]

Classification Report:
              precision    recall  f1-score   support

      Normal       0.86      1.00      0.93        25
      Attack       1.00      0.43      0.60         7

    accuracy                           0.88        32
   macro avg       0.93      0.71      0.76        32
weighted avg       0.89      0.88      0.85        32



In [85]:
# Show predictions for the first 10 test nodes
for i in range(min(20,len(test_x))):
    node=test_x[i].item()
    
    actual=int(data.y[node].item())
    probability = probabilities[node].item()
    predicted = int(predictions[node].item())

    print(
        f"Node: {node}, "
        f"Actual: {actual}, "
        f"Attack probability: {probability:.4f}, "
        f"Predicted: {predicted}"
    )

Node: 112, Actual: 0, Attack probability: 0.2651, Predicted: 0
Node: 94, Actual: 0, Attack probability: 0.1962, Predicted: 0
Node: 130, Actual: 0, Attack probability: 0.2476, Predicted: 0
Node: 1, Actual: 0, Attack probability: 0.4200, Predicted: 0
Node: 29, Actual: 0, Attack probability: 0.4501, Predicted: 0
Node: 123, Actual: 0, Attack probability: 0.4128, Predicted: 0
Node: 12, Actual: 1, Attack probability: 0.7578, Predicted: 1
Node: 106, Actual: 0, Attack probability: 0.1892, Predicted: 0
Node: 53, Actual: 0, Attack probability: 0.4262, Predicted: 0
Node: 107, Actual: 0, Attack probability: 0.2035, Predicted: 0
Node: 144, Actual: 0, Attack probability: 0.2773, Predicted: 0
Node: 128, Actual: 1, Attack probability: 0.4274, Predicted: 0
Node: 34, Actual: 1, Attack probability: 0.6492, Predicted: 1
Node: 129, Actual: 1, Attack probability: 0.3571, Predicted: 0
Node: 28, Actual: 0, Attack probability: 0.2607, Predicted: 0
Node: 72, Actual: 0, Attack probability: 0.2468, Predicted: 0
N